# 🛡️ SAFER FDS: AI Engine v1
**Smart AI Fraud Evaluation & Risk Intelligence**

Notebook ini mendemonstrasikan pipeline end-to-end Machine Learning, Explainable AI (SHAP), dan Graph Intelligence v1 untuk **SAFER FDS** (Fraud Detection System). Notebook ini siap dipindahkan (copy-paste) langsung ke Kaggle maupun Google Colab.

### Arsitektur AI FDS
1. **Data Ingestion & Preprocessing**: Memuat data transaksi sintetis Indonesia dan melakukan rekayasa fitur.
2. **Risk Scoring Engine (XGBoost & LightGBM)**: Melatih model klasifikasi biner untuk mendeteksi transaksi Fraud vs Non-Fraud.
3. **Explainable AI (SHAP)**: Mengidentifikasi parameter kontribusi terbesar pemicu fraud dan membuat *Indonesian Audit Reasoning Generator* otomatis.
4. **Graph Intelligence (NetworkX)**: Memetakan jaringan sindikat *Mule Rings* dan *Device Sharing Farms* menggunakan analisis topologi graf (PageRank & Degree Centrality).
5. **LLM Co-Pilot**: Demonstrasi pembuatan prompt otomatis untuk LLM untuk menyusun laporan penyelidikan AML (Anti-Money Laundering).


In [ ]:
# Install library yang diperlukan di environment Kaggle / Colab
!pip install -q xgboost lightgbm shap networkx matplotlib seaborn pandas numpy scikit-learn


## 1. Import Libraries & Load Datasets
Langkah pertama adalah memuat pustaka Python yang diperlukan serta file dataset `train_transactions.csv` and `test_transactions.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import xgboost as xgb
import lightgbm as lgb
import shap
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]


In [ ]:
# Memuat dataset
try:
    train_df = pd.read_csv("train_transactions.csv")
    test_df = pd.read_csv("test_transactions.csv")
    print(f"Train Dataset Loaded: {train_df.shape[0]} baris, {train_df.shape[1]} kolom")
    print(f"Test Dataset Loaded: {test_df.shape[0]} baris, {test_df.shape[1]} kolom")
except FileNotFoundError:
    print("File CSV tidak ditemukan! Pastikan Anda telah menjalankan script generate_dataset.py terlebih dahulu")


## 2. Exploratory Data Analysis (EDA)
Mari kita visualisasikan rasio kelas fraud, sebaran nominal transaksi (amount), serta korelasi antar parameter indikator kecurangan.


In [ ]:
# 1. Distribusi Kelas Target (Fraud vs Normal)
fraud_counts = train_df['is_fraud'].value_counts()
fraud_pct = train_df['is_fraud'].value_counts(normalize=True) * 100

print("=== Sebaran Transaksi Fraud ===")
for val, count in fraud_counts.items():
    label = "FRAUD (1)" if val == 1 else "NORMAL (0)"
    print(f"{label}: {count} rows ({fraud_pct[val]:.2f}%)")

sns.countplot(x='is_fraud', data=train_df, palette='viridis')
plt.title('Rasio Transaksi Normal vs Fraud')
plt.xlabel('Status Transaksi (0: Normal, 1: Fraud)')
plt.ylabel('Jumlah Transaksi')
plt.show()


In [ ]:
# 2. Distribusi Nominal Transaksi untuk Fraud vs Normal (Log Scale)
plt.figure(figsize=(12, 5))
sns.kdeplot(train_df[train_df['is_fraud'] == 0]['amount'], label='Normal (0)', fill=True, color='teal')
sns.kdeplot(train_df[train_df['is_fraud'] == 1]['amount'], label='Fraud (1)', fill=True, color='crimson')
plt.xscale('log')
plt.title('Distribusi Nominal Transaksi (Skala Logaritma IDR)')
plt.xlabel('Jumlah Nominal (IDR)')
plt.ylabel('Density')
plt.legend()
plt.show()


In [ ]:
# 3. Korelasi antar Indikator Fraud
indicator_cols = [
    'is_velocity_anomaly', 'is_geo_mismatch', 'is_off_hours', 
    'is_high_value_for_rail', 'is_suspicious_ip', 'is_risky_merchant', 
    'is_new_account', 'has_failed_attempts', 'is_device_mismatch', 
    'is_sim_swap', 'is_unusual_beneficiary', 'is_fraud'
]
corr = train_df[indicator_cols].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Matriks Korelasi Indikator Sinyal Risiko & Target Fraud')
plt.show()


## 3. Data Preprocessing & Feature Engineering
Mengubah data kategori menjadi numerik menggunakan Label Encoding dan melakukan standarisasi data nominal transaksi.


In [ ]:
def preprocess_data(df, label_encoders=None, scaler=None, is_train=True):
    processed = df.copy()
    
    # Kolom kategori yang perlu di-encode
    categorical_cols = [
        'sender_bank', 'receiver_bank', 'payment_rail', 
        'ewallet_provider', 'merchant', 'merchant_category', 'channel', 
        'device_type', 'device_brand'
    ]
    
    if is_train:
        label_encoders = {}
        for col in categorical_cols:
            le = LabelEncoder()
            processed[col] = le.fit_transform(processed[col].astype(str))
            label_encoders[col] = le
    else:
        for col in categorical_cols:
            le = label_encoders[col]
            # Tangani kategori baru di test set
            processed[col] = processed[col].astype(str).map(lambda s: s if s in le.classes_ else le.classes_[0])
            processed[col] = le.transform(processed[col])
            
    # Standardisasi nominal numerik
    numerical_cols = ['amount', 'account_age_days', 'velocity_count', 'geo_distance_km']
    
    if is_train:
        scaler = StandardScaler()
        processed[numerical_cols] = scaler.fit_transform(processed[numerical_cols])
    else:
        processed[numerical_cols] = scaler.transform(processed[numerical_cols])
        
    # Hapus kolom ID dan string non-prediktif
    drop_cols = [
        'id', 'timestamp', 'sender_name', 'sender_account', 'sender_city', 'sender_province',
        'receiver_name', 'receiver_account', 'receiver_city', 'receiver_province',
        'ip_address', 'device_fingerprint'
    ]
    
    if 'is_fraud' in processed.columns:
        y = processed['is_fraud']
        X = processed.drop(columns=drop_cols + ['is_fraud'], errors='ignore')
    else:
        y = None
        X = processed.drop(columns=drop_cols, errors='ignore')
        
    return X, y, label_encoders, scaler

# Eksekusi Preprocessing
X_train, y_train, encoders, scaler = preprocess_data(train_df, is_train=True)
X_test, y_test, _, _ = preprocess_data(test_df, label_encoders=encoders, scaler=scaler, is_train=False)

print(f"Fitur Training: {X_train.shape}")
print(f"Fitur Testing: {X_test.shape}")


## 4. Model Training & Evaluation (XGBoost & LightGBM)
Kita akan melatih dua model algoritma klasifikasi ensemble Gradient Boosting terbaik: **XGBoost** dan **LightGBM**.


In [ ]:
# 1. Model Training: XGBoost
print("Melatih Model XGBoost Classifier...")
scale_pos = (len(y_train) - sum(y_train)) / sum(y_train) # Mengatasi ketidakseimbangan kelas
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

# Prediksi
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]


In [ ]:
# 2. Model Training: LightGBM
print("Melatih Model LightGBM Classifier...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)

# Prediksi
y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]


In [ ]:
# Evaluasi Performa Model
print("\n=== EVALUASI PERFORMA MODEL XGBOOST ===")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.4f}")

print("\n=== EVALUASI PERFORMA MODEL LIGHTGBM ===")
print(classification_report(y_test, y_pred_lgb))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lgb):.4f}")


In [ ]:
# Plotting ROC & Precision-Recall Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test, y_prob_lgb)
ax1.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {roc_auc_score(y_test, y_prob_xgb):.4f})', color='dodgerblue', lw=2)
ax1.plot(fpr_lgb, tpr_lgb, label=f'LightGBM (AUC = {roc_auc_score(y_test, y_prob_lgb):.4f})', color='darkorange', lw=2)
ax1.plot([0, 1], [0, 1], 'k--', label='Tebakan Acak')
ax1.set_title('Receiver Operating Characteristic (ROC) Curve')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend()

# Precision-Recall Curve
p_xgb, r_xgb, _ = precision_recall_curve(y_test, y_prob_xgb)
p_lgb, r_lgb, _ = precision_recall_curve(y_test, y_prob_lgb)
ax2.plot(r_xgb, p_xgb, label=f'XGBoost (AP = {average_precision_score(y_test, y_prob_xgb):.4f})', color='dodgerblue', lw=2)
ax2.plot(r_lgb, p_lgb, label=f'LightGBM (AP = {average_precision_score(y_test, y_prob_lgb):.4f})', color='darkorange', lw=2)
ax2.set_title('Precision-Recall Curve')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend()
plt.show()


## 5. Explainable AI (SHAP) & Auto-Reasoning Generator
Di industri finansial, penjelasan *mengapa* AI mencurigai suatu transaksi sangat penting untuk kepatuhan hukum (*auditability*). Kita menggunakan **SHAP** untuk mengidentifikasi pemicu utama kenaikan risiko, lalu menerjemahkannya secara otomatis ke narasi analisis berbahasa Indonesia.


In [ ]:
# Hitung SHAP Values menggunakan TreeExplainer pada subset data uji untuk efisiensi waktu
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test.iloc[:500])

# Visualisasi pengaruh fitur secara keseluruhan
shap.summary_plot(shap_values, X_test.iloc[:500], max_display=12)


In [ ]:
def generate_indonesian_reasoning(tx_row, original_tx, explainer_model):
    row_df = pd.DataFrame([tx_row])
    shap_val = explainer_model(row_df)
    
    # Gabung nama fitur dengan kontribusinya
    feat_contribs = list(zip(row_df.columns, shap_val.values[0], row_df.iloc[0]))
    # Urutkan dari pengaruh positif terbesar (yang menaikkan skor fraud)
    feat_contribs_sorted = sorted(feat_contribs, key=lambda x: x[1], reverse=True)
    
    # Pilih 3 fitur dengan kontribusi positif tertinggi
    top_pos_features = [f for f in feat_contribs_sorted if f[1] > 0.05][:3]
    
    # Hitung skor probabilitas prediksi model ML
    prob = xgb_model.predict_proba(row_df)[:, 1][0]
    score = int(prob * 100)
    
    # Kamus terjemahan fitur ke bahasa Indonesia yang ramah analis bank
    feature_desc_mapping = {
        'is_velocity_anomaly': "anomali kecepatan/frekuensi transaksi dalam waktu singkat (velocity)",
        'is_geo_mismatch': "jarak koordinat antar-kota tidak wajar (geo mismatch)",
        'is_off_hours': "waktu transaksi di luar jam kerja (dini hari/off-hours)",
        'is_high_value_for_rail': "nominal transaksi melampaui batas wajar metode pembayaran",
        'is_suspicious_ip': "alamat IP terindikasi dalam threat intelligence feed",
        'is_risky_merchant': "pembayaran ke merchant kategori berisiko (Gambling/Crypto/Lending)",
        'is_new_account': "umur akun nasabah di bawah 30 hari (new account)",
        'has_failed_attempts': "terjadi kegagalan autentikasi sandi/OTP berulang",
        'is_device_mismatch': "akses menggunakan sidik jari device berbeda (device mismatch)",
        'is_sim_swap': "pergantian kartu SIM nomor terdaftar pada device baru",
        'is_unusual_beneficiary': "penerima dana baru pertama kali ditransfer (unusual beneficiary)",
        'velocity_count': "tingginya aktivitas transaksi beruntun",
        'geo_distance_km': "jarak geografis pengirim-penerima"
    }
    
    reasons = []
    for feat, contrib, val in top_pos_features:
        desc = feature_desc_mapping.get(feat, feat)
        reasons.append(f"**{desc}**")
        
    severity = "CRITICAL" if score >= 80 else "HIGH" if score >= 60 else "MEDIUM" if score >= 35 else "LOW"
    action = {
        "CRITICAL": "Auto-block transaksi, freeze sementara akun pengirim, dan eskalasi ke tim Fraud Operations.",
        "HIGH": "Hold dana transaksi, kirim verifikasi biometrik wajah ke HP nasabah.",
        "MEDIUM": "Log transaksi dan tingkatkan monitoring akun selama 7 hari ke depan.",
        "LOW": "Izinkan transaksi berjalan normal dan catat di log audit."
    }[severity]
    
    narration = f"### 🛡️ SAFER AI Reasoning (XAI Report) \n"
    narration += f"- **Status Risiko**: `{severity}`\n"
    narration += f"- **Risk Score**: `{score}/100` | **Probabilitas Fraud**: `{prob:.2%}`\n\n"
    
    if reasons:
        narration += f"**Hasil Analisis Kontribusi Fitur (SHAP):**\n"
        narration += f"Transaksi dicurigai karena kontribusi utama dari: {', '.join(reasons[:-1])} serta {reasons[-1]}.\n\n"
    else:
        narration += f"**Hasil Analisis Kontribusi Fitur (SHAP):**\nTransaksi ini dinilai aman dan selaras dengan pola normal nasabah.\n\n"
        
    narration += f"**Rekomendasi Tindakan Analis AML:**\n> {action}"
    return narration

# Uji coba penjelasan pada transaksi fraud yang ada di dataset
fraud_indices = test_df[test_df['is_fraud'] == 1].index
if len(fraud_indices) > 0:
    idx = fraud_indices[0]
    raw_tx = test_df.iloc[idx]
    proc_tx = X_test.iloc[idx]
    print(generate_indonesian_reasoning(proc_tx, raw_tx, explainer))
else:
    print("Tidak ada transaksi fraud di subset data uji")


## 6. Graph Intelligence: Mule Ring & Device Sharing Farm Detection
Sindikat kejahatan (seperti judi online / pencucian uang) biasanya menggunakan banyak akun (*Mule Ring*) atau berpindah-pindah login pada perangkat fisik yang sama (*Device Farm*). Pola hubungan non-linear ini tidak terdeteksi oleh tabel data biasa. Kita membangun graf transaksi menggunakan **NetworkX** untuk mendeteksinya.


In [ ]:
# Membangun Directed Graph (Graf Berarah)
G = nx.DiGraph()

# Masukkan transaksi sebagai Edges (sisi) dan Akun sebagai Nodes (titik)
for idx, row in test_df.iterrows():
    sender = f"{row['sender_bank']}_{row['sender_account']}"
    receiver = f"{row['receiver_bank']}_{row['receiver_account']}"
    
    if sender == receiver: continue # Hindari transfer ke diri sendiri
    
    G.add_edge(
        sender, 
        receiver, 
        amount=row['amount'],
        timestamp=row['timestamp'],
        is_fraud=row['is_fraud'],
        device_fingerprint=row['device_fingerprint']
    )

print(f"Graf berhasil dibangun dengan {G.number_of_nodes()} Akun (Nodes) dan {G.number_of_edges()} Aliran Dana (Edges).")


In [ ]:
# 1. PageRank Centrality: Menemukan Akun Hub/Penampung Utama Sindikat
pagerank = nx.pagerank(G)
top_hubs = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]

print("=== Top 5 Akun dengan PageRank Tertinggi (Kandidat Hub Penampung/Mule Hub) ===")
for node, rank in top_hubs:
    print(f"Akun: {node} | PageRank Score: {rank:.6f} | Total Transfer Diterima: {G.in_degree(node)}")


In [ ]:
# 2. Algoritma Deteksi Mule Ring Sindikat (Many Senders -> 1 Receiver -> Outflow)
mule_hubs_detected = []
for node in G.nodes():
    in_edges = G.in_edges(node)
    unique_senders = set([u for u, v in in_edges])
    
    # Jika akun menerima transfer dari >= 4 rekening unik dan mentransfernya keluar
    if len(unique_senders) >= 4:
        out_edges = G.out_edges(node, data=True)
        if len(out_edges) >= 1:
            mule_hubs_detected.append((node, len(unique_senders), list(out_edges)))

print(f"Terdeteksi {len(mule_hubs_detected)} klaster terindikasi sindikat Mule Ring.")
for hub, m_count, outs in mule_hubs_detected[:2]:
    print(f"\n[!] Mule Hub Account: {hub} (Menerima dari {m_count} akun)")
    for u, v, d in outs:
        print(f"    -> Transfer Keluar: ke Rekening {v} | Nominal: Rp {d['amount']:,} | Tanggal: {d['timestamp']}")


In [ ]:
# 3. Algoritma Deteksi Device Sharing Farm
from collections import defaultdict
device_to_accs = defaultdict(set)

for idx, row in test_df.iterrows():
    acc = f"{row['sender_bank']}_{row['sender_account']}"
    device_to_accs[row['device_fingerprint']].add(acc)

shared_devs = {dev: accs for dev, accs in device_to_accs.items() if len(accs) >= 5}
print(f"Terdeteksi {len(shared_devs)} Perangkat Fisik (Device Fingerprint) yang dishare secara massal (>=5 akun):")
for dev, accs in list(shared_devs.items())[:2]:
    print(f"- Device fingerprint: {dev} digunakan login oleh {len(accs)} akun: {list(accs)[:4]}...")


In [ ]:
# 4. Visualisasi Sub-Graf Sindikat Mule Ring
if mule_hubs_detected:
    hub_node = mule_hubs_detected[0][0]
    sub_nodes = {hub_node}
    
    for u, v in G.in_edges(hub_node): sub_nodes.add(u)
    for u, v in G.out_edges(hub_node): sub_nodes.add(v)
    
    sub_G = G.subgraph(sub_nodes)
    
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(sub_G, seed=42)
    
    colors = []
    for n in sub_G.nodes():
        if n == hub_node:
            colors.append('crimson') # Hub
        elif G.out_edges(n):
            colors.append('orange') # Mule
        else:
            colors.append('lightblue') # Penerima akhir
            
    nx.draw_networkx_nodes(sub_G, pos, node_color=colors, node_size=1000, alpha=0.9)
    nx.draw_networkx_edges(sub_G, pos, width=2.0, alpha=0.6, edge_color='gray', arrowsize=20)
    nx.draw_networkx_labels(sub_G, pos, font_size=8, font_weight="bold")
    
    plt.title(f"Visualisasi Hubungan Sindikat Mule Ring (Collector Hub: {hub_node})")
    plt.axis('off')
    plt.show()
else:
    print("Tidak ada klaster Mule Ring yang ditemukan untuk divisualisasikan.")


## 7. LLM Co-Pilot Integration (Mock Prompt Builder)
Sebagai asisten pintar (Co-Pilot), SAFER FDS mengirimkan ringkasan data transaksi beserta kontribusi SHAP ke model LLM untuk membuat draf investigasi resmi AML secara instan.


In [ ]:
def build_llm_copilot_prompt(tx, probability, top_features):
    prompt = f"""
=== PROMPT INPUT UNTUK LLM CO-PILOT ===
[ROLE]
Anda adalah Senior AML (Anti-Money Laundering) Analyst dan Co-Pilot AI untuk SAFER FDS.

[TUGAS]
Susun draf laporan audit resmi (2-3 paragraf) dalam Bahasa Indonesia untuk dianalisis oleh analis manusia di Audit Queue.

[DATA TRANSAKSI]
- ID Transaksi: {tx['id']}
- Pengirim: {tx['sender_name']} ({tx['sender_bank']}, {tx['sender_city']})
- Penerima/Merchant: {tx['merchant']} ({tx['merchant_category']})
- Nominal Transfer: Rp {tx['amount']:,}
- Metode Pembayaran: {tx['payment_rail']}
- Device: {tx['device_brand']} (Baru: {'Ya' if tx['is_new_device'] else 'Tidak'})
- Alamat IP: {tx['ip_address']}

[Sinyal ML & SHAP]
- Probabilitas Fraud: {probability:.2%}
- Pemicu Utama (SHAP): {', '.join(top_features)}

[FORMAT OUTPUT]
Paragraf 1: Analisis modus kejahatan finansial yang paling cocok (contoh: Jaringan Mule, Pengambilalihan Akun, atau Social Engineering).
Paragraf 2: Temuan teknis pemicu (anomali geografi/travel, anomali perangkat keras, atau jam kerja).
Paragraf 3: Rekomendasi tindakan pembekuan dana / verifikasi kepatuhan hukum bank.
"""
    return prompt

# Ambil contoh data
sample_tx = test_df.iloc[0]
sample_features = ["impossible travel (lintas kota dalam waktu singkat)", "device fingerprint baru", "transfer ke merchant judi online"]
print(build_llm_copilot_prompt(sample_tx, 0.9234, sample_features))


## 8. Exporting Models & Preprocessors for Local Deployment
Untuk menjalankan model ini secara lokal atau menghubungkannya ke backend FastAPI, kita harus mengekspor bobot model (`xgb_model.json`, `lgb_model.txt`) serta objek preprocessor (`scaler.pkl`, `label_encoders.pkl`). File ini akan berada di direktori output `/kaggle/working/` dan dapat diunduh langsung.


In [ ]:
import joblib

# 1. Simpan Model XGBoost
xgb_model.save_model("xgb_model.json")
print("Saved XGBoost model as: xgb_model.json")

# 2. Simpan Model LightGBM
lgb_model.booster_.save_model("lgb_model.txt")
print("Saved LightGBM model as: lgb_model.txt")

# 3. Simpan Preprocessor (Label Encoders & Scaler)
joblib.dump(encoders, "label_encoders.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Saved preprocessors: label_encoders.pkl, scaler.pkl")
